Attempts to implement damped Newton method

In [146]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


Nikolay's two distance functions. Function embedded_distance will compute the distance between two tensors, even with different leg dimensions, by extending tensors by zeros. Function embedded_distance_with_additional_sign_fixing does the same, but additionally applies relative discrete gauge fixing. The code is suboptimal, but we need these functions only once in this computation.

In [3]:
function embedded_distance(tens1, tens2)
	if tens1.qhape != tens2.qhape
		throw("nonequal qhapes, code is not suited for such a situation.")
	end
	extended_shape = max.(tens1.shape, tens2.shape)
	tens1_ext = extend_blocks_by_zeros(tens1, extended_shape)
	tens2_ext = extend_blocks_by_zeros(tens2, extended_shape)
	tens1_ext - tens2_ext |> norm
end

function embedded_distance_with_additional_sign_fixing(tens1, tens2)
	if tens1.qhape != tens2.qhape
		throw("nonequal qhapes, code is not suited for such a situation.")
	end
	extended_shape = max.(tens1.shape, tens2.shape)
	tens1_ext = (extend_blocks_by_zeros(tens1, extended_shape) |> ju_to_py)
	tens2_ext = (extend_blocks_by_zeros(tens2, extended_shape) |> ju_to_py)
	tens2_f, tens2_accepted_elements, _ = fix_discrete_gauge(tens2_ext)
	tens1_f, _ = fix_discrete_gauge(tens1_ext, tens2_accepted_elements)
	tens1_f.to_ndarray() - tens2_f.to_ndarray() |> norm
end;

Now, we retrieve the trajectory corresponding to the found approximation of the critical temperature.

In [97]:
gilt_eps = 1e-4
chi = 10
cg_eps = 1e-10
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 0,
	"rotate" => true,
)
Jratio = 1.0

relT=1.0
#do 3 steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, 3, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[4], accepted_elements, _ = fix_discrete_gauge(traj[4]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

We fixed the gauge along the trajectory and normalize tensors. Note that it is not always possible to fix discrete gauge uniformly as at the beginning of the trajectory, the shape of the tensor changes. This may have caused some warnings, but it is fine.

We can now compute the distances. For the first ten tensors, we will use additional relative gauge fixing. We still will see warnings because of the changing shape of the tensor at the beginning of the trajectory. However, this does not introduce any issues, especially at the part of the trajectory close to the fixed point tensor.

In [98]:
distances = Float64[]
for i in 1:length(traj)-1
	if i <= 10
		push!(distances, embedded_distance_with_additional_sign_fixing(traj[i], traj[i+1]))
	else
		try
			push!(distances, embedded_distance(traj[i], traj[i+1]))
		catch
			push!(distances, NaN)
		end
	end
end

println(distances)

[0.4103426470854554, 0.27053354343413843, 0.04353867475300184]


┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.08304579896222142 and became 0.0. Index CartesianIndex(3, 4, 1, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.08304579896220585 and became 0.0. Index CartesianIndex(1, 1, 4, 3) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.07230561794605667 and became 0.0. Index CartesianIndex(3, 2, 3, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.0723056179460413 and became 0.0. Index CartesianIndex(1, 3, 2, 3) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -0.03249633820171291 and became

In [99]:
function fix_discrete_gauge(A::Z2Tensor; tol = 1e-7)
    A1py, accepted_elements, _ = fix_discrete_gauge(ju_to_py(A); tol = tol)
    return py_to_ju(A1py), accepted_elements
end

fix_discrete_gauge (generic function with 5 methods)

In [142]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

In [147]:
A[1] = traj[4]
for i in 1:20
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    RA = gilt(A[i], accepted_elements[i], gilt_pars);
    println("i=",i) 
    println("||R(A[i])-A[i]||= ", embedded_distance(RA, A[i]))
    flush(stdout)
    #include("./Lab/newton-step-SR.jl");
    deltaA[i] = newton_correction(A[i], 5, accepted_elements[i], gilt_pars);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    A[i+1] = A[i] + deltaA[i]
end

i=1
||R(A[i])-A[i]||= 0.03024339787576557


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (4.631851566752148e-46, 6.8392482115900175e-31, 4.076026290304671e-31, 4.789859569287356e-21, 7.842315827853482e-16, 6.436145830725834e-16, 6.436145830725834e-16, 3.1519932519551483e-16, 3.1519932519551483e-16, 3.0622087901550225e-14, 3.0622087901550225e-14)
└ *  number of operations = 43


||deltaA[i]||= 0.024915709935144446
i=2
||R(A[i])-A[i]||= 0.005850793060283604


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (2.2147222472380718e-45, 1.169569490016272e-30, 4.881621456358689e-30, 4.1149089890355318e-19, 1.7799453049018493e-17, 6.245268789995831e-16, 6.245268789995831e-16, 1.8524353803257087e-15, 1.8524353803257087e-15, 1.2019678053322162e-15, 1.2019678053322162e-15)
└ *  number of operations = 43


||deltaA[i]||= 0.004836378436804552
i=3
||R(A[i])-A[i]||= 0.00165780910480079


...

||deltaA[i]||= 6.320094076791412e-12
i=19
||R(A[i])-A[i]||= 2.7320511702570257e-12


┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  5 eigenvalues converged
│ *  norm of residuals = (4.813914133834058e-35, 7.493408271061919e-22, 1.8114615006574945e-21, 1.2828021212826456e-13, 4.40508489609553e-13)
└ *  number of operations = 34


||deltaA[i]||= 3.2033736960696396e-12
i=20
||R(A[i])-A[i]||= 7.038065773230541e-13


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (6.372434931054268e-47, 8.726523631174337e-31, 2.2617894206911605e-30, 4.0966213563705863e-17, 4.557285194660109e-18, 1.212508707397625e-15, 1.212508707397625e-15, 8.133535455821447e-16, 8.133535455821447e-16, 6.8454069228857655e-16, 6.8454069228857655e-16)
└ *  number of operations = 43


||deltaA[i]||= 7.064514638118823e-13


In [148]:
include("./Lab/newton-step-SR.jl");
deltaA1 = newton_correction(A[1], 5, accepted_elements[1], gilt_pars);

GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  5 eigenvalues converged
│ *  norm of residuals = (3.027495963983217e-35, 1.603330423275423e-22, 8.055802593474995e-23, 2.9487220642280986e-14, 4.699351098300212e-13)
└ *  number of operations = 34


EIGENVALUES (INITIAL):
1.9870691501489048 + 0.0im
-0.9162160360072799 + 0.0im
-0.9034101124751837 + 0.0im
0.4422878554999291 + 0.0im
-0.3363884937530674 + 0.0im


In [149]:
include("./Lab/newton-step-SR.jl");
deltaA1 = newton_correction(A[1], 5, accepted_elements[1], gilt_pars);

GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (4.7422739720738984e-46, 2.0731449600820682e-32, 2.425001591233105e-31, 4.0507791731254924e-21, 1.607884123191493e-16, 6.25270570855062e-16, 6.25270570855062e-16, 6.332559277674941e-16, 6.332559277674941e-16, 1.575235589712753e-15, 1.575235589712753e-15)
└ *  number of operations = 43


EIGENVALUES (INITIAL):
1.975551937303975 + 0.0im
-0.9162196191400006 + 0.0im
-0.8941923945265601 + 0.0im
0.44217298692110313 + 0.0im
-0.32324924655698345 + 0.0im
-0.0004425658581109253 + 0.3179718463442044im
-0.0004425658581109253 - 0.3179718463442044im
-0.07033561499059392 + 0.3011891414371625im
-0.07033561499059392 - 0.3011891414371625im
0.06619915529317486 + 0.2887107092919767im
0.06619915529317486 - 0.2887107092919767im


In [132]:
println("norm of deltaA0: ", norm(deltaA0))
A1 = A0 + deltaA0
A1, accepted_elements1 = fix_discrete_gauge(A1; tol = 1e-7);
A2 = gilt(A1, accepted_elements1, gilt_pars);
println("A1-A2 distance: ", embedded_distance(A2, A1))

norm of deltaA0: 0.02466225273066161
A1-A2 distance: 0.006615641543326471


In [133]:
deltaA1 = newton_correction(A1, 5, accepted_elements1, gilt_pars);

┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  5 eigenvalues converged
│ *  norm of residuals = (6.8657140749627725e-34, 8.465261153062205e-22, 2.428343001335635e-22, 4.877703381471613e-15, 5.2889099662269e-13)
└ *  number of operations = 34


EIGENVALUES (INITIAL):
2.0052748115752212 + 0.0im
-0.9357197327124224 + 0.0im
-0.9305333908082051 + 0.0im
0.44375912054577293 + 0.0im
-0.35394305199694576 + 0.0im


In [134]:
println("norm of deltaA1: ", norm(deltaA1))
A2 = A1 + deltaA1
A2, accepted_elements2 = fix_discrete_gauge(A2; tol = 1e-7);
A3 = gilt(A2, accepted_elements2, gilt_pars);
println("A3-A2 distance: ", embedded_distance(A3, A2))

norm of deltaA1: 0.005164638688492864
A3-A2 distance: 0.00142952183646763
